## Reconstruction evaluation on a test set

Companion to `Good_reconstruction_visualization`, but quantitative and over the
**whole test set** instead of a single scenario. Evaluates a trained GridFM
model on the masked node-feature reconstruction task in two modes:

- **Point** — one deterministic forward pass (dropout off): RMSE / MAE per
  feature, split by bus type (PQ / PV / REF).
- **Probabilistic** — MC-dropout (`NUM_MC_SAMPLES` stochastic passes) giving a
  per-bus predictive mean ± std: adds NLL, PICP (interval coverage), MPIW
  (sharpness) and a calibration curve.

All metrics are on the **masked** entries (the reconstruction target) and
**denormalized** to physical units (MW, MVar, p.u., degrees).

The metric/plot logic lives in `gridfm_graphkit.utils.evaluation` so the script
`examples/evaluate_reconstruction.py` and this notebook share one source.

In [ ]:
from pathlib import Path

import torch
import yaml

from gridfm_graphkit.datasets.powergrid_datamodule import LitGridDataModule
from gridfm_graphkit.io.param_handler import NestedNamespace
from gridfm_graphkit.tasks.feature_reconstruction_task import FeatureReconstructionTask
from gridfm_graphkit.utils import evaluation as ev
from gridfm_graphkit.utils.visualization import visualize_error, visualize_quantity_heatmap
from gridfm_graphkit.datasets.globals import PD, QD, PG, QG, VM, VA

# Find the repo root (works whether the notebook runs from repo root or here).
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

CONFIG = REPO / "examples/config/case30_ieee_base.yaml"
DATA = REPO / "examples/data"
MODEL = REPO / "examples/models/GridFM_v0_2.pth"
NETWORK_IDX = 0          # which test network (multi-net configs)
NUM_MC_SAMPLES = 20      # MC-dropout passes for the probabilistic pass
DEVICE = "cpu"           # MPS lacks the scatter op torch_geometric needs on Apple Silicon
MASKED_ONLY = True       # score only reconstructed entries (the task)

## Load config, data module and trained model

In [ ]:
with open(CONFIG) as f:
    cfg = NestedNamespace(**yaml.safe_load(f))
cfg.data.workers = 0  # single-process loading: avoids spawn issues, fine for a test set
torch.manual_seed(cfg.seed)

dm = LitGridDataModule(cfg, str(DATA))
dm.setup("test")
test_loader = dm.test_dataloader()
normalizer = dm.node_normalizers[NETWORK_IDX]

task = FeatureReconstructionTask(cfg, dm.node_normalizers, dm.edge_normalizers)
task.load_state_dict(torch.load(MODEL, map_location=DEVICE))
task = task.to(DEVICE).eval()
print(f"Network: {cfg.data.networks[NETWORK_IDX]}  |  test batches: {len(test_loader[NETWORK_IDX])}")

## Qualitative check on a single scenario

Same view as `Good_reconstruction_visualization` — a sanity check before the
aggregate numbers. `output` here is the raw (normalized) model output; the
visualization helpers denormalize internally.

In [ ]:
batch = next(iter(test_loader[NETWORK_IDX])).to(DEVICE)
with torch.no_grad():
    output = task(
        x=batch.x, pe=batch.pe, edge_index=batch.edge_index,
        edge_attr=batch.edge_attr, batch=batch.batch, mask=batch.mask,
    )
visualize_error(batch, output, normalizer)
visualize_quantity_heatmap(batch, output, VM, "Voltage magnitude", "p.u.", normalizer)
visualize_quantity_heatmap(batch, output, VA, "Voltage Angle", "degrees", normalizer)

## Run inference over the full test set

One pass collects both the deterministic point prediction and (if
`NUM_MC_SAMPLES >= 2`) the MC-dropout mean + std for every node.

In [ ]:
arrays = ev.run_inference(
    task, test_loader[NETWORK_IDX], normalizer,
    device=DEVICE, num_mc_samples=NUM_MC_SAMPLES,
)
{k: v.shape for k, v in arrays.items()}

## Point predictions — RMSE / MAE per feature and bus type

In [ ]:
point = ev.point_metrics(arrays, masked_only=MASKED_ONLY)
point

In [ ]:
ev.plot_pred_vs_true(arrays, which="point", masked_only=MASKED_ONLY);

## Probabilistic predictions — uncertainty & calibration

- **NLL** — Gaussian negative log-likelihood (lower is better).
- **PICP@95%** — fraction of true values inside the 95% interval; well-calibrated
  ≈ 0.95. Below → intervals too narrow (over-confident); above → too wide.
- **MPIW** — mean 95% interval width (sharpness; smaller is better *given* coverage).
- **sharpness** — mean predictive std.

In [ ]:
prob = ev.prob_metrics(arrays, masked_only=MASKED_ONLY)
prob

### Calibration curve

Observed vs nominal coverage. On the diagonal = calibrated; below = the model is
over-confident (intervals too tight), above = under-confident.

In [ ]:
ev.plot_calibration(arrays, masked_only=MASKED_ONLY);

### Error vs uncertainty

Does a larger predictive std actually flag larger errors? Points hugging the
diagonal mean the uncertainty is informative.

In [ ]:
ev.plot_error_vs_uncertainty(arrays, masked_only=MASKED_ONLY);

In [ ]:
ev.plot_pred_vs_true(arrays, which="mc", masked_only=MASKED_ONLY);

## Save the metric tables (optional)

In [ ]:
out = REPO / "eval_out"
out.mkdir(exist_ok=True)
net = cfg.data.networks[NETWORK_IDX]
point.to_csv(out / f"{net}_point_metrics.csv")
prob.to_csv(out / f"{net}_prob_metrics.csv")
print("saved to", out)